# Feature Selection

_Feature Engineering_

**Throw away features that do not help, so the model is cheaper, faster, and overfits less.**

More features is not more knowledge. Past a point, extra columns add noise, cost, and overfitting &mdash; the model fits quirks of the training sample instead of the real pattern. The goal of selection is a small subset that keeps the signal and drops the rest.

       The hard part is how you judge a feature. The three families are three answers, trading off speed against faithfulness to the real model:

---

This notebook works through feature selection one step at a time. Run each cell, read the note above it, and watch how dropping noise columns shrinks the train-test gap — then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Bolt 200 noise columns onto real data

We take the real breast-cancer data (30 genuinely informative features) and concatenate 200 columns of pure Gaussian noise that carry no signal at all. A fixed random seed keeps the numbers reproducible. This sets up the overfitting trap: a classifier handed all 230 columns will eagerly fit the noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X_real, y = data.data, data.target          # X_real: 569 rows x 30 real features
n_real = X_real.shape[1]                     # = 30
n_noise = 200                                # number of junk columns to add

rng = np.random.default_rng(0)               # fixed seed -> same numbers every run
X_noise = rng.normal(size=(X_real.shape[0], n_noise))   # pure noise, no signal
X = np.hstack([X_real, X_noise])             # 569 rows x 230 columns (30 real + 200 junk)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

### Step 2 — Score every feature's informativeness

Mutual information measures how much each column tells us about the label. Computed on the training rows only, the 30 real features should tower over the 200 noise columns, which sit near zero. The bar chart (left of a two-panel figure) makes the signal-vs-noise split obvious, with a red line marking where the real features end.

In [ ]:
mi = mutual_info_classif(X_tr, y_tr, random_state=0)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
colors = ["seagreen"] * n_real + ["lightgray"] * n_noise
ax[0].bar(range(X.shape[1]), mi, color=colors, width=1.0)
ax[0].axvline(n_real - 0.5, color="red", ls="--", lw=1)
ax[0].set_title("STEP 2: feature score (green=30 real, gray=200 noise)")
ax[0].set_xlabel("feature index")
ax[0].set_ylabel("mutual information with y")

### Step 3 — Reproduce the overfitting problem

Now fit the model on all 230 raw features. With 200 noise columns to latch onto, the classifier memorizes quirks of the training sample: the training accuracy stays high but the test accuracy drops, and the gap between them blows up — the classic signature of overfitting. We record the train score, the test score, and their gap.

In [ ]:
raw_model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000)),
])
raw_model.fit(X_tr, y_tr)

raw_train = accuracy_score(y_tr, raw_model.predict(X_tr))
raw_test = accuracy_score(y_te, raw_model.predict(X_te))
raw_gap = raw_train - raw_test

### Step 4 — Select features and show the fix

The fix is `SelectKBest` placed *inside* a `Pipeline`, so the selection is re-fit on each training fold and never peeks at the test rows (no leakage). We sweep `k` to watch test accuracy rise and then plateau as we keep the real features, then commit to `k = 30` (the number of genuinely informative columns). The same model, now fed only signal, recovers test accuracy and the train-test gap shrinks.

In [ ]:
def make_selected_model(k):
    return Pipeline([
        ("select", SelectKBest(score_func=mutual_info_classif, k=k)),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000)),
    ])

ks = [2, 5, 10, 20, 30, 50, 100, 230]        # number of features kept
test_curve = []
for k in ks:
    m = make_selected_model(k)
    m.fit(X_tr, y_tr)                        # SelectKBest fit on TRAIN only
    test_curve.append(accuracy_score(y_te, m.predict(X_te)))

ax[1].plot(ks, test_curve, "o-", color="steelblue")
ax[1].set_title("STEP 4: test accuracy vs #features kept (rises, then plateaus)")
ax[1].set_xlabel("k = features kept")
ax[1].set_ylabel("test accuracy")
ax[1].set_xscale("log")
plt.tight_layout()
plt.show()

# Commit to k = n_real (30), the number of genuinely informative columns.
fix_model = make_selected_model(k=n_real)    # keep top 30
fix_model.fit(X_tr, y_tr)
fix_train = accuracy_score(y_tr, fix_model.predict(X_tr))
fix_test = accuracy_score(y_te, fix_model.predict(X_te))
fix_gap = fix_train - fix_test

print(f"PROBLEM (all 230 raw):  train={raw_train:.3f}  test={raw_test:.3f}  gap={raw_gap:.3f}")
print(f"FIX (top {n_real} selected):   train={fix_train:.3f}  test={fix_test:.3f}  gap={fix_gap:.3f}")
print(f"PROBLEM (raw): {raw_test:.3f}   →   FIX (engineered): {fix_test:.3f}")

## Reference implementation — scikit-learn

### Step 1 — Filter: score each feature alone, keep the top k

The cheapest family scores every feature independently against the target and keeps the highest. Here a chi-square test ranks the columns (breast-cancer features are all non-negative, which chi-square requires) and we keep the top 10. The selector lives in a pipeline so cross-validation re-fits it per fold; we also fit it on all data once, just to *look* at which features score highest.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

# The book demonstrates selection on high-dimensional text (bag-of-words / tf-idf).
# Same three families here on a bundled tabular dataset.
# Full notebooks: github.com/alicezheng/feature-engineering-book
X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# ---------- (1) FILTER: score each feature alone, keep the top k ----------
# chi2 needs non-negative features (breast-cancer features are all >= 0).
filter_pipe = Pipeline([
    ("filter", SelectKBest(chi2, k=10)),   # univariate scores vs the target
    ("clf",    LogisticRegression(max_iter=5000)),
])
print("filter  (SelectKBest chi2, k=10):",
      round(cross_val_score(filter_pipe, X, y, cv=cv).mean(), 3))

# Inspect which features the filter kept (fit on all data only to LOOK):
skb = SelectKBest(chi2, k=10).fit(X, y)
print("  top-10 chi2 scores:", np.round(np.sort(skb.scores_)[::-1][:10], 1))

### Step 2 — Wrapper: recursive feature elimination around the model

A wrapper uses the model itself to judge features. `RFE` trains the model, drops the single weakest feature, refits, and repeats until only `k` remain. This is more faithful than a filter (it sees the model's own preferences) but more expensive, because it retrains many times. We standardize first so the coefficients are comparable.

In [ ]:
# ---------- (2) WRAPPER: recursive feature elimination around the model ----------
# RFE trains the model, drops the weakest feature, refits, repeats down to k.
wrapper_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("rfe",   RFE(LogisticRegression(max_iter=5000),
                  n_features_to_select=10, step=1)),
    ("clf",   LogisticRegression(max_iter=5000)),
])
print("wrapper (RFE, k=10):",
      round(cross_val_score(wrapper_pipe, X, y, cv=cv).mean(), 3))

### Step 3 — Embedded: L1 (Lasso) sparsity selects during training

An embedded method gets selection "for free" during a single fit. An L1 (Lasso) penalty drives unhelpful coefficients to exactly zero; a larger `alpha` zeroes out more of them. The features with non-zero weights are the ones the model chose to keep — we read them straight off the fitted coefficient vector.

In [ ]:
# ---------- (3) EMBEDDED: L1 (Lasso) sparsity selects during training ----------
# A larger alpha (lambda) zeroes out more coefficients -> fewer features kept.
lasso = Pipeline([("scale", StandardScaler()),
                  ("lasso", Lasso(alpha=0.05))]).fit(X, y)
coef = lasso.named_steps["lasso"].coef_
kept = np.flatnonzero(coef)              # non-zero weights == selected features
print("embedded (Lasso L1): kept %d of %d features" % (kept.size, X.shape[1]))
print("  selected feature indices:", kept.tolist())

## Visualize the data & results

_On real data, how many features do you actually need? A FILTER (mutual information with the target) ranks the features; we keep the top-k and watch accuracy. The curve plateaus fast: a handful of features gets almost all the accuracy._

### Step 1 — Build a filter-inside-cross-validation sweep

On the *real* 30-feature data (no added noise), we ask how many features we actually need. A mutual-information filter ranks the columns and keeps the top `k`. The selector sits inside the pipeline so it is re-fit on each training fold — no held-out rows leak into the ranking. We sweep `k` and record the cross-validated accuracy for each.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

X, y = load_breast_cancer(return_X_y=True)   # 569 rows, 30 features
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# A FILTER ranks features by mutual information with the target; keep the top k.
# The selector lives INSIDE the pipeline, so it is re-fit on the training fold
# of every split -> no leakage of held-out rows into the ranking.
def mi(Xx, yy):
    return mutual_info_classif(Xx, yy, random_state=0)

ks = [1, 2, 3, 5, 8, 12, 20, 30]
accs = []
for k in ks:
    pipe = make_pipeline(
        StandardScaler(),
        SelectKBest(mi, k=k),                # filter: top-k by mutual information
        LogisticRegression(max_iter=5000),
    )
    accs.append(round(cross_val_score(pipe, X, y, cv=cv).mean(), 3))

### Step 2 — Read off the accuracy-vs-k curve

Printing accuracy against `k` shows the curve plateaus fast: a single feature already gets you into the low 0.9s, and a handful captures almost all the accuracy that all 30 give. The practical lesson is that much of the feature set is redundant — you can keep a small subset with little loss.

In [ ]:
for k, a in zip(ks, accs):
    print("k=%-3d  accuracy=%.3f" % (k, a))
# k=1 0.917 ... k=8 0.954 ... k=30 0.979  -> a few features get most of the accuracy.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a tf-idf matrix with 40,000 columns and need a quick first prune before training anything. Which family do you reach for, and what is the one number it cannot see?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the scale: 40,000 features means anything that trains a model per feature is too slow for a first pass. — _Wrappers train a model many times; that is impractical at 40,000 columns for a quick cut._
- Use a FILTER: score each column by chi-square or mutual information against the label and keep the top k. — _A filter costs one cheap statistic per feature and sorts &mdash; fast even at this width._
- Remember its blind spot: it scores each feature alone, so it cannot see INTERACTIONS between features. — _Two columns useless apart but predictive together would both be dropped by a filter._

**Answer:** Use a filter method (e.g. SelectKBest with chi2 or mutual information). It is the only family cheap enough for a fast first cut on 40,000 features. Its blind spot: it judges each feature on its own, so it cannot see feature interactions.

</details>

**Problem 2.** A colleague runs SelectKBest on the entire dataset, THEN splits into train and test, and is thrilled by the test accuracy. What did they do wrong, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot that the selector saw the test rows: the whole dataset, including test, was used to choose the features. — _Information from the held-out rows leaked into the feature choice, inflating the test score._
- Name it: this is data leakage. The test set is no longer truly unseen. — _Any step fit on data that includes the test set makes the score optimistic._
- Move the selector INSIDE a Pipeline and run it inside cross-validation, fitting on the training fold only. — _Then selection is re-done per fold on training data alone, so the held-out fold stays genuinely unseen._

**Answer:** They leaked: selecting on the whole dataset let the test rows influence which features were kept, so the test accuracy is optimistic. Fix: wrap the selector in a Pipeline and select inside cross-validation on the training fold only, never on the full data before the split.

</details>

**Problem 3.** You want feature selection to come "for free" as part of training a linear model, and you want unhelpful features driven to exactly zero. What do you use, and why does it produce exact zeros where ridge would not?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Choose an EMBEDDED method: a linear model with an L1 (Lasso) penalty. — _Embedded selection happens during the single fit; no separate scoring or subset search is needed._
- Recall the geometry: the L1 budget region is a diamond with corners on the axes; the loss contours often touch a corner. — _A corner is a point where some weights are exactly zero &mdash; those features are dropped._
- Contrast with L2 (ridge): its budget region is a smooth circle with no corners, so it shrinks weights but rarely zeroes them. — _Ridge keeps every feature (just smaller), so it does not select; Lasso does._

**Answer:** Use Lasso (an L1-penalized linear model) &mdash; selection is embedded in training. L1's diamond-shaped budget has corners on the axes, so the loss contours frequently touch at a point where some weights are exactly zero, dropping those features. L2 (ridge) has a smooth circular budget with no corners, so it only shrinks weights and never zeroes them.

</details>